# Verificar la entrega en Colab y estimar el score

Comprueba el repo de la entrega con el **harness oficial** en las condiciones de la evaluación (Colab con CPU, 1 hilo, 3 ejecuciones independientes) y estima el score.

**Qué hace:**
1. Clona los materiales oficiales y descarga los datos.
2. Clona el repo de la entrega como lo harán los organizadores: desde un clon limpio, instalando su `requirements.txt`.
3. **test_demo** (`--source file:data/test_demo.npz`, 3 ejecuciones): score, acierto, recall por clase y setup.
4. **Latencia** (sobre train, 3 ejecuciones de 60 s de `predict`, como la evaluación oficial): media, mediana, p99, máximo y timeouts.
5. **Score estimado** para la evaluación oficial: $\text{Score} \approx \sqrt{N/8}\cdot \text{gm}_f$, con $N = \min(10^7,\ 60\,\text{s}/t)$.

**Uso:** *Entorno de ejecución → Cambiar tipo → CPU*, y después *Ejecutar todo*. Si el repo es privado, la celda 2 te pedirá un token de GitHub. Tarda unos 10–15 min.

## 1. Configuración

In [1]:
REPO_ENTREGA = "https://github.com/diegoarangureen/gdg-ml-contest-entrega.git"
RUNS = 3                 # ejecuciones independientes, como la evaluación oficial
TIME_BUDGET_S = 60       # presupuesto de predict por ejecución (oficial: 60 s)
GM_F_VAL_CORR = 0.520    # gm_f de la validación corrompida de #30 (14 250 muestras × 4 versiones), medida en local: la referencia fiable

import os, subprocess, sys, json, glob, platform
MATERIALES = "/content/ml-contest-materials"
ENTREGA = "/content/submission-check"

## 2. Materiales oficiales, datos y clon limpio de la entrega

In [3]:
if not os.path.isdir(MATERIALES):
    subprocess.run(["git", "clone", "-q", "https://github.com/GDG-UAM/ml-contest-materials.git", MATERIALES], check=True)
for f in ("train_100k.npz", "test_demo.npz"):
    ruta = f"{MATERIALES}/data/{f}"
    if not os.path.exists(ruta) or os.path.getsize(ruta) < 1000:
        subprocess.run(["curl", "-sL", "--create-dirs", "-o", ruta,
                        f"https://github.com/GDG-UAM/ml-contest-materials/releases/latest/download/{f}"], check=True)
    print(f"{f}: {os.path.getsize(ruta) / 1e6:.1f} MB")

if not os.path.isdir(ENTREGA):
    url = REPO_ENTREGA
    r = subprocess.run(["git", "clone", "-q", url, ENTREGA], capture_output=True, text=True)
    if r.returncode != 0:  # repo privado: se pide un token (no se guarda en el notebook)
        from getpass import getpass
        token = getpass("Repo privado. Token de GitHub con acceso de lectura: ")
        url = REPO_ENTREGA.replace("https://", f"https://x-access-token:{token}@")
        subprocess.run(["git", "clone", "-q", url, ENTREGA], check=True)
        subprocess.run(["git", "-C", ENTREGA, "remote", "set-url", "origin", REPO_ENTREGA], check=True)
print(sorted(os.listdir(ENTREGA)))
assert os.path.exists(f"{ENTREGA}/model.py"), "falta model.py en la raíz del repo"
if os.path.exists(f"{ENTREGA}/requirements.txt"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{ENTREGA}/requirements.txt"], check=True)

import numpy, numba
print(f"Python {platform.python_version()} | numpy {numpy.__version__} | numba {numba.__version__}")
print(subprocess.run("lscpu | grep -E 'Model name|^CPU\\(s\\)'", shell=True, capture_output=True, text=True).stdout)

train_100k.npz: 413.2 MB
test_demo.npz: 1.7 MB
Repo privado. Token de GitHub con acceso de lectura: ··········
['.git', '.gitignore', 'README.md', 'model.py', 'modelo_numba.npz', 'motor_multi.py', 'requirements.txt']
Python 3.13.18 | numpy 2.4.6 | numba 0.64.0
CPU(s):                                  2
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz



## 3. Harness

Cada llamada ejecuta `harness.py evaluate` desde los materiales oficiales, igual que la evaluación. Los resultados se guardan en `/content/resultados_entrega/`.

- **test_demo:** 400 muestras «como las del test». La ejecución termina al agotarlas, así que su score (~3–4) es mucho menor que el oficial: sirve para el acierto y el recall.
- **Latencia (train):** 60 s de `predict`. El fichero de train se agota a las 100 000 muestras, así que con un modelo rápido termina antes (`stream_exhausted`). El score de esta ejecución **no** es el real (es la distribución de train); solo interesa la latencia.

In [4]:
def harness(nombre, extra):
    salida = f"/content/resultados_entrega/{nombre}"
    cmd = [sys.executable, "harness.py", "evaluate", "--model", f"{ENTREGA}/model.py", "--runs", str(RUNS),
           "--out", salida, "--set", f"time_budget_s={TIME_BUDGET_S}", *extra]
    print("$", " ".join(cmd[1:]))
    r = subprocess.run(cmd, cwd=MATERIALES, capture_output=True, text=True)
    print("\n".join(l for l in (r.stdout + r.stderr).splitlines() if "run" in l or "WARN" in l.upper() or "error" in l.lower())[-3000:])
    runs = [json.load(open(p)) for p in sorted(glob.glob(f"{salida}/*/run_*.json"))]
    assert runs, f"el harness no generó resultados:\n{r.stdout[-2000:]}\n{r.stderr[-2000:]}"
    return runs

TEST = harness("test_demo", ["--source", "file:data/test_demo.npz"])
TRAIN = harness("latencia_train", [])

$ harness.py evaluate --model /content/submission-check/model.py --runs 3 --out /content/resultados_entrega/test_demo --set time_budget_s=60 --source file:data/test_demo.npz
rank  project   mean   best  worst  median  trimmed_mean  mean_minus_std  failed_runs  mean_accuracy_all  mean_n_scored  mean_latency_s
  [model] run 0  score=3.488  n=400  acc=0.825  latency=0.1ms  end=stream_exhausted  ok
  [model] run 1  score=3.488  n=400  acc=0.825  latency=0.1ms  end=stream_exhausted  ok
  [model] run 2  score=3.488  n=400  acc=0.825  latency=0.2ms  end=stream_exhausted  ok
$ harness.py evaluate --model /content/submission-check/model.py --runs 3 --out /content/resultados_entrega/latencia_train --set time_budget_s=60
rank  project   mean   best  worst  median  trimmed_mean  mean_minus_std  failed_runs  mean_accuracy_all  mean_n_scored  mean_latency_s
  [model] run 0  score=64.767  n=100000  acc=0.876  latency=0.1ms  end=stream_exhausted  ok
  [model] run 1  score=64.767  n=100000  acc=0.876  

## 4. Resultados y score estimado

In [5]:
import numpy as np
import pandas as pd

def gm_f(rec, suelo=1e-4):
    rec = np.asarray(rec, float)
    return float(np.exp(np.mean(np.log(np.maximum((1.33 * rec - 0.33) * rec ** 2, suelo)))))

def fila(r, fase):
    lat = r.get("latency_s") or {}
    return {"fase": fase, "run": r["run"], "estado": r["status"], "fin": r["end_reason"], "n": r["n_scored"],
            "score": r["score"], "acierto": r["accuracy_all"], "setup (s)": r["setup_time_s"],
            "lat. media (µs)": lat.get("mean", np.nan) * 1e6, "lat. mediana (µs)": lat.get("median", np.nan) * 1e6,
            "lat. p99 (µs)": lat.get("p99", np.nan) * 1e6, "lat. máx (µs)": lat.get("max", np.nan) * 1e6,
            "timeouts": r["n_soft_timeout"] + r["n_hard_timeout"], "excepciones": r["n_exception"] + r["n_invalid"]}

tabla = pd.DataFrame([fila(r, "test_demo") for r in TEST] + [fila(r, "latencia (train)") for r in TRAIN])
display(tabla.round(4))

rec = np.mean([r["per_class_accuracy"] for r in TEST], axis=0)
display(pd.DataFrame([rec], columns=[f"R{k}" for k in range(8)], index=["recall test_demo"]).round(3))

t = float(np.mean([r["latency_s"]["mean"] for r in TRAIN]))        # media de las 3 ejecuciones de latencia
N = min(1e7, 60.0 / t)
gm_test = gm_f(rec)
print(f"\nLatencia media en Colab: {t * 1e6:.1f} µs  →  N ≈ {N:,.0f} muestras en 60 s")
print(f"gm_f test_demo: {gm_test:.3f} | gm_f validación corrompida (local): {GM_F_VAL_CORR:.3f}")
print(f"Score estimado (gm_f val-corr, la fiable): {np.sqrt(N / 8) * GM_F_VAL_CORR:.1f}")
print(f"Score estimado (gm_f test_demo, 400 muestras, más ruidosa): {np.sqrt(N / 8) * gm_test:.1f}")

,fase,run,estado,fin,n,score,acierto,setup (s),lat. media (µs),lat. mediana (µs),lat. p99 (µs),lat. máx (µs),timeouts,excepciones
0,test_demo,0,ok,stream_exhausted,400,3.4881,0.8250,19.3558,142.1375,135.002,243.4348,284.066,0,0
1,test_demo,1,ok,stream_exhausted,400,3.4881,0.8250,1.9434,132.6266,128.680,245.1955,291.482,0,0
2,test_demo,2,ok,stream_exhausted,400,3.4881,0.8250,1.7165,178.8956,181.217,316.3981,636.132,0,0
3,latencia (train),0,ok,stream_exhausted,100000,64.7668,0.8765,2.3914,139.7263,131.899,239.5354,6007.323,0,0
4,latencia (train),1,ok,stream_exhausted,100000,64.7668,0.8765,1.6452,139.1165,132.060,229.8754,2723.531,0,0
5,latencia (train),2,ok,stream_exhausted,100000,64.7668,0.8765,1.7458,141.3827,134.278,236.6113,2385.143,0,0


,R0,R1,R2,R3,R4,R5,R6,R7
recall test_demo,0.82,0.98,0.8,0.68,0.98,1.0,0.64,0.7



Latencia media en Colab: 140.1 µs  →  N ≈ 428,341 muestras en 60 s
gm_f test_demo: 0.493 | gm_f validación corrompida (local): 0.520
Score estimado (gm_f val-corr, la fiable): 120.3
Score estimado (gm_f test_demo, 400 muestras, más ruidosa): 114.1


## 5. Comprobaciones de las reglas

In [6]:
todos = TEST + TRAIN
chk = {
    "Las 6 ejecuciones terminan bien (status ok)": all(r["status"] == "ok" for r in todos),
    "Setup < 120 s en todas": all((r["setup_time_s"] or 1e9) < 120 for r in todos),
    "Sin excepciones ni salidas inválidas": all(r["n_exception"] + r["n_invalid"] == 0 for r in todos),
    "Sin timeouts (0,5 s soft / 0,75 s hard)": all(r["n_soft_timeout"] + r["n_hard_timeout"] == 0 for r in todos),
    "Latencia máxima < 0,5 s": all(r["latency_s"]["max"] < 0.5 for r in todos if r.get("latency_s")),
    "Sin abstenciones": all(r["n_abstain"] == 0 for r in todos),
    "Acierto test_demo idéntico en las 3 ejecuciones": len({round(r["accuracy_all"], 6) for r in TEST}) == 1,
}
for k, v in chk.items():
    print(("✅" if v else "❌"), k)
print("\nReferencia local de #30: acierto test_demo 0,825 | score test_demo 3,488 | setup en frío ~14–22 s | Colab ~170 µs")

✅ Las 6 ejecuciones terminan bien (status ok)
✅ Setup < 120 s en todas
✅ Sin excepciones ni salidas inválidas
✅ Sin timeouts (0,5 s soft / 0,75 s hard)
✅ Latencia máxima < 0,5 s
✅ Sin abstenciones
✅ Acierto test_demo idéntico en las 3 ejecuciones

Referencia local de #30: acierto test_demo 0,825 | score test_demo 3,488 | setup en frío ~14–22 s | Colab ~170 µs
